# LLM-assisted interpretation of the KH Coder outputs (Section 2.9)

This notebook performs the auxiliary interpretation step described in Section 2.9 of the manuscript
*From matrix to materials: Mapping five decades of elastin research through multivariate text mining*.

It gives three outputs of the KH Coder analyses to a large language model in **one API request** and keeps a
complete record of that request:

| Saved as | Content |
|---|---|
| `llm/prompts/instructions.txt`, `llm/prompts/input.txt` | the two texts sent to the model, exactly as sent |
| `llm/inputs/` | the data files from which the input text was built, unchanged |
| `llm/outputs/response_raw.json` | the response body as received from the API |
| `llm/outputs/output.md` | the text of the response, unedited |
| `llm/RUN.md`, `llm/run_log.jsonl` | date, endpoint, model, parameters, token counts and checksums for every request sent |

The model does not receive the abstracts and performs no quantitative analysis. Its output is supplementary
information: the authors check every statement against the KH Coder outputs before using it.

Run the cells in order. Read the prompt shown by step 4 before running step 5. Step 5 is the only cell that
calls the API, and it asks for confirmation before it sends anything.

In [ ]:
# 1. Setup
%pip install -q "openai==3.26.1"

import csv, datetime, getpass, hashlib, io, json, os, pathlib, platform, re, shutil, urllib.error, urllib.request, zoneinfo
import openai
from openai import OpenAI

if openai.__version__ != "3.26.1":      # the version this notebook was tested with
    raise RuntimeError(f"openai {openai.__version__} is loaded instead of 3.26.1: restart the runtime and run this cell again.")
print("python", platform.python_version(), "| openai", openai.__version__)

## 2. Settings

All settings are in the next cell; the wording of the prompt is in step 4. `RUN.md` records what was actually sent.

In [ ]:
# 2. Settings
MODEL = "gpt-5.2-2025-12-11"    # the snapshot named in Section 2.9
REASONING_EFFORT = "none"       # the default of this model; it accepts a temperature only with "none"
TEMPERATURE = 0                 # None = do not send (API default)
MAX_OUTPUT_TOKENS = 6000

# What the analyzed texts are (first sentence of the prompt)
CORPUS = ("the abstracts of 19,125 elastin-related publications retrieved from the Web of Science "
          "(publication years 1975 to 2025)")

# Input files: derived data in the public repository, read at a fixed commit
REPO = "Doc-Koba/elastin-textmining"
COMMIT = "1166178f264e293ac838ffa430ffb4e65070af26"
INPUT_FILES = {
    "terms": "data/terms120.csv",                                   # the 120 terms (data of Table S1)
    "network_nodes": "data/network_nodes.csv",                      # co-occurrence network of Figure 2: terms
    "network_edges": "data/network_edges.csv",                      # co-occurrence network of Figure 2: edges
    "period_terms": "data/ca_characteristic_terms_monkin_ja.txt",   # characteristic terms of each period
}
LOCAL_DATA_DIR = None           # path of a local clone of the repository, to read the files from disk instead

# Numbers reported in the manuscript. Step 3 stops if the input files do not agree with them.
EXPECTED = {"terms": 120, "clusters": 7, "terms in the network": 44, "edges": 60, "subgraphs": 11,
            "terms in the correspondence analysis": 60, "periods": 6}

OUT_DIR = pathlib.Path("llm")
LOCAL_TZ = "Asia/Tokyo"         # time zone of the date in RUN.md (UTC is recorded as well)
ASK_BEFORE_SENDING = True       # step 5 waits for the word SEND
ALLOW_RERUN = False             # the manuscript reports a single run; see step 5

# Checks of the settings (nothing is sent here)
zoneinfo.ZoneInfo(LOCAL_TZ)
if not LOCAL_DATA_DIR and not re.fullmatch(r"[0-9a-f]{40}", COMMIT):
    raise ValueError("COMMIT must be the full 40-character commit SHA, so that the input files are fixed.")
if not (isinstance(MAX_OUTPUT_TOKENS, int) and MAX_OUTPUT_TOKENS > 0):
    raise ValueError("MAX_OUTPUT_TOKENS must be a positive whole number.")
if TEMPERATURE is not None and not isinstance(TEMPERATURE, (int, float)):
    raise ValueError("TEMPERATURE must be a number or None.")
if REASONING_EFFORT is not None and not isinstance(REASONING_EFFORT, str):
    raise ValueError("REASONING_EFFORT must be a string or None.")
print("settings ok:", MODEL, "| reasoning effort", REASONING_EFFORT, "| temperature", TEMPERATURE)

## 3. Input files

The files are read into memory and checked against the numbers in `EXPECTED`. Step 5 copies them unchanged to
`llm/inputs/`.

In [ ]:
# 3. Input files
def fetch(path):
    """Bytes of one input file, from a local clone or from GitHub at the fixed commit."""
    if LOCAL_DATA_DIR:
        return pathlib.Path(LOCAL_DATA_DIR, path).read_bytes()
    url = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/{path}"
    try:
        with urllib.request.urlopen(url, timeout=60) as r:
            return r.read()
    except urllib.error.URLError as e:
        raise RuntimeError(f"Could not download {url} ({e}).") from None

def as_text(raw):
    """UTF-8 text with Unix line endings, without empty lines at the start and end."""
    return raw.decode("utf-8-sig").replace("\r\n", "\n").replace("\r", "\n").strip("\n")

def table(label, required):
    rows = list(csv.DictReader(io.StringIO(text[label])))
    missing = [c for c in required if not rows or c not in rows[0]]
    if missing:
        raise ValueError(f"{INPUT_FILES[label]}: no data rows or missing column(s) {missing}")
    return rows

def whole(row, column, label):
    try:
        return int(row[column])
    except ValueError:
        raise ValueError(f"{INPUT_FILES[label]}: '{row[column]}' in column {column} is not a whole number") from None

input_source = f"local folder {LOCAL_DATA_DIR}" if LOCAL_DATA_DIR else f"https://github.com/{REPO} at commit {COMMIT}"
raw_inputs, text, sha256 = {}, {}, {}
for label, path in INPUT_FILES.items():
    raw_inputs[path] = fetch(path)
    sha256[path] = hashlib.sha256(raw_inputs[path]).hexdigest()
    text[label] = as_text(raw_inputs[path])

terms = table("terms", ["term", "frequency", "cluster", "subgraph_figure2", "used_in_figure3"])
nodes = table("network_nodes", ["term", "subgraph", "frequency"])
edges = table("network_edges", ["source", "target", "subgraph_source", "subgraph_target", "within_subgraph"])
periods = re.findall(r"^\*「(\d{4}-\d{4})」", text["period_terms"], flags=re.M)

# Consistency between the files
cluster_of = {t["term"]: t["cluster"] for t in terms}
subgraph_of = {t["term"]: t["subgraph_figure2"] for t in terms}
for t in terms:
    whole(t, "cluster", "terms")
    whole(t, "frequency", "terms")
for n in nodes:
    if n["term"] not in cluster_of:
        raise ValueError(f"{INPUT_FILES['network_nodes']}: term '{n['term']}' is not in {INPUT_FILES['terms']}")
    if str(whole(n, "subgraph", "network_nodes")) != subgraph_of[n["term"]].lstrip("0"):
        raise ValueError(f"'{n['term']}': subgraph {n['subgraph']} in the network file "
                         f"but '{subgraph_of[n['term']]}' in the term file")
node_terms = {n["term"] for n in nodes}
claimed = {t["term"] for t in terms if t["subgraph_figure2"]}
if claimed != node_terms:
    raise ValueError(f"Terms with a subgraph in {INPUT_FILES['terms']} and terms in {INPUT_FILES['network_nodes']} "
                     f"differ: {sorted(claimed ^ node_terms)}")
for e in edges:
    if e["source"] not in node_terms or e["target"] not in node_terms:
        raise ValueError(f"{INPUT_FILES['network_edges']}: edge {e['source']}-{e['target']} names a term "
                         "that is not in the network file")

found = {
    "terms": len(terms),
    "clusters": len({t["cluster"] for t in terms}),
    "terms in the network": len(nodes),
    "edges": len(edges),
    "subgraphs": len({n["subgraph"] for n in nodes}),
    "terms in the correspondence analysis": sum(t["used_in_figure3"] == "yes" for t in terms),
    "periods": len(periods),
}
print("source:", input_source)
for name, value in found.items():
    print(f"  {name}: {value}" + ("" if EXPECTED.get(name) == value else f"   <-- EXPECTED says {EXPECTED.get(name)}"))
if found != EXPECTED:
    raise ValueError("The input files do not agree with EXPECTED (step 2). Check the files, or correct EXPECTED "
                     "if the numbers in the manuscript have changed.")
print("periods:", ", ".join(periods))

## 4. Prompt

The model receives two texts: short instructions, and one input text with the data and the task. The data are
shown as three tables and one list. The tables are the repository files with their rows sorted (Table 1 by
cluster, Table 2 by subgraph) and with the cluster number of each term added to Tables 2 and 3 from Table 1; the
list is the software output, unchanged apart from line endings. The prompt contains no cluster labels, no domain names and no conclusions
of the authors.

In [ ]:
# 4. Prompt
def to_csv(rows, columns):
    buffer = io.StringIO()
    writer = csv.DictWriter(buffer, fieldnames=columns, lineterminator="\n", extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)
    return buffer.getvalue().strip("\n")

table_1 = to_csv(sorted(terms, key=lambda t: (int(t["cluster"]), -int(t["frequency"]))), list(terms[0].keys()))
table_2 = to_csv([{**n, "cluster": cluster_of[n["term"]]} for n in sorted(nodes, key=lambda n: int(n["subgraph"]))],
                 ["term", "subgraph", "cluster", "frequency"])
table_3 = to_csv([{**e, "cluster_source": cluster_of[e["source"]], "cluster_target": cluster_of[e["target"]]} for e in edges],
                 list(edges[0].keys()) + ["cluster_source", "cluster_target"])

INSTRUCTIONS = (
    "You assist researchers in interpreting the results of a text-mining analysis. "
    "Base every statement on the data given in the user message. "
    "To suggest what a group of terms may represent, you may use general knowledge of the research field, "
    "but mark every such suggestion as an interpretation. "
    "Do not introduce specific facts that are not in the data: no numbers, dates, names of people or organizations, "
    "and no references."
)

USER_INPUT = f"""The data below are outputs of a text-mining analysis carried out with KH Coder on {CORPUS}. After a stop-word list had been applied, the {found['terms']} most frequent nouns and adjectives were selected and analyzed in three ways:

(A) Hierarchical cluster analysis (Jaccard distance, Ward's method). The dendrogram was cut into {found['clusters']} clusters.
(B) Co-occurrence network (Jaccard coefficient). Only the {found['edges']} strongest co-occurrences were kept as edges; {found['terms in the network']} of the terms have at least one edge, and a modularity-based community detection assigned these terms to {found['subgraphs']} subgraphs.
(C) Correspondence analysis of terms by publication period ({found['periods']} periods: {', '.join(periods)}). It used the {found['terms in the correspondence analysis']} terms whose frequencies differed most between the periods (largest chi-square values). For each period, the software Monkin Reporting for KH Coder then listed the characteristic terms.

TABLE 1. The {found['terms']} terms, sorted by cluster and by frequency within each cluster (source: {INPUT_FILES['terms']}).
Columns: frequency = total number of occurrences in the abstracts; rank = rank by frequency among the {found['terms']} terms; cluster = cluster number in (A); subgraph_figure2 = subgraph number in (B), empty if the term has no edge; used_in_figure3 = yes if the term was used in (C).
<table_1>
{table_1}
</table_1>

TABLE 2. The {found['terms in the network']} terms of the co-occurrence network, sorted by subgraph (source: {INPUT_FILES['network_nodes']}; the column cluster was added from Table 1).
<table_2>
{table_2}
</table_2>

TABLE 3. The {found['edges']} edges of the co-occurrence network (source: {INPUT_FILES['network_edges']}; the columns cluster_source and cluster_target were added from Table 1). within_subgraph = no marks an edge between two different subgraphs.
<table_3>
{table_3}
</table_3>

LIST 4. Characteristic terms of each period (source: {INPUT_FILES['period_terms']}; unedited output of the software, in Japanese). The first block lists the terms near the origin of the plot, which are not characteristic of any period; each of the following blocks lists the terms characteristic of one period.
<list_4>
{text['period_terms']}
</list_4>

TASK
Propose possible interpretations of these results in four parts.

Part 1. Research themes of the clusters. For each of the {found['clusters']} clusters, the research theme or themes that its terms may represent.
Part 2. Relationships between research domains. How the themes of Part 1 relate to one another, and whether some of them belong together as a broader research domain, judging from the co-occurrence network (terms of different clusters in the same subgraph; edges between clusters or between subgraphs).
Part 3. Research trends by period. Which terms characterize each period, whether and how they differ between periods, and what this may indicate about research trends.
Part 4. Integrated interpretation. A short overall interpretation that combines Parts 1 to 3.

RULES
- Write numbered statements, one claim per statement, numbered consecutively through all four parts (S1, S2, ...).
- After the number, mark each statement as [data] if it only restates or summarizes the data, or as [interpretation] if it goes beyond them.
- End each statement of Parts 1 to 3 with its evidence in parentheses: the cluster, subgraph or period, followed by the terms on which the statement rests. End each statement of Part 4 with the numbers of the statements it combines.
- If the data do not support a clear theme, relationship or difference, say so instead of proposing one.
- Do not give numbers that are not in the data, and do not cite publications or authors.
- Write in English, in no more than 1,500 words.
"""

prompt_built_from = {"source": input_source, "sha256": dict(sha256)}

print(f"instructions: {len(INSTRUCTIONS)} characters | input: {len(USER_INPUT)} characters, {USER_INPUT.count(chr(10))} lines")
print("\n--- instructions ---\n" + INSTRUCTIONS)
print("\n--- input (tables and list shortened here; step 5 saves the full text) ---")
shown, in_block, kept = [], False, 0
for line in USER_INPUT.splitlines():
    if line.startswith("</"):
        in_block = False
    if in_block:
        kept += 1
        if kept == 4:
            shown.append("  ...")
        if kept >= 4:
            continue
    if line.startswith("<") and not line.startswith("</"):
        in_block, kept = True, 0
    shown.append(line)
print("\n".join(shown))

## 5. The API request

This cell sends **one** request and records it. The response body is written to disk and logged as soon as it
arrives, before anything else is done with it. A request that returns no response (refused by the API, or failed)
is logged as well, and so is a request interrupted while waiting for the answer. The record covers the requests sent from this working folder; it cannot know about requests
sent from elsewhere.

If a response already exists, the cell stops. Set `ALLOW_RERUN = True` only to send another request on purpose:
all files of earlier runs are kept, `RUN.md` lists every run, and Section 2.9 can then no longer say
"in a single run".

In [ ]:
# 5. The API request (the only cell that calls the API)
def utc_stamp(moment):
    return moment.strftime("%Y-%m-%dT%H:%M:%SZ")

def save(name, data):
    path = OUT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data if isinstance(data, bytes) else data.encode("utf-8"))

def save_record(entries):
    """Rewrite run_log.jsonl and RUN.md from the list of all requests sent from this folder."""
    save("run_log.jsonl", "".join(json.dumps(e, ensure_ascii=False) + "\n" for e in entries))
    got = [e for e in entries if e["kind"] == "response"]
    lost = [e for e in entries if e["kind"] == "no_response"]
    md = ["# Run record: LLM-assisted interpretation (Section 2.9)", "",
          "Written by `llm_interpretation.ipynb`. It lists every request that the notebook sent from one working "
          "folder; `run_log.jsonl` holds the same record in machine-readable form. Checksums are SHA-256 of the "
          "files as the notebook wrote them (UTF-8, LF line endings).", "",
          f"- Requests that returned a response: {len(got)}",
          f"- Requests that returned no response: {len(lost)}", ""]
    for e in got:
        f = e["files"]
        md += [f"## Run {e['run']}" + ("" if e["usable"] else " (no usable output)"), "",
               f"- Date and time of the request: {e['sent_local']} ({e['sent_utc']})",
               f"- Endpoint: {e['endpoint']}; client openai-python {e['openai_python']} on Python {e['python']}; "
               "automatic retries disabled",
               f"- Model requested: `{e['model_requested']}`"]
        if e["parsed"]:
            md.append(f"- Model reported by the API: `{e['model_returned']}`"
                      + ("" if e["model_returned"] == e["model_requested"] else " (differs from the name requested)"))
        md.append("- Request body: `model`, `instructions`, `input` and "
                  + "; ".join(f"{k} = {json.dumps(v)}" for k, v in e["parameters_sent"].items())
                  + ". Nothing else was sent: no tools and no earlier conversation.")
        if e["parsed"]:
            t = e["tokens"]
            md += ["- Settings reported back by the API: "
                   + "; ".join(f"{k} = {json.dumps(v)}" for k, v in e["settings_reported_by_api"].items()),
                   f"- Response: id `{e['response_id']}`, request id `{e['request_id']}`, status `{e['status']}`"
                   + (f" (reason: {e['incomplete_reason']})" if e["incomplete_reason"] else ""),
                   "- Tokens: " + ("not reported" if t is None else
                                   f"input {t['input']}, output {t['output']} (reasoning {t['reasoning']}), total {t['total']}")]
        else:
            md.append(f"- Response: request id `{e['request_id']}`; {e.get('note', 'not read yet')}")
        md += [f"- Texts sent: `{f['instructions']}` (SHA-256 `{e['prompt_sha256']['instructions']}`), "
               f"`{f['input']}` (SHA-256 `{e['prompt_sha256']['input']}`)",
               f"- Data files from which the input text was built ({e['input_source']}), copied unchanged to `{f['inputs']}/`:",
               *[f"  - `{path}` (SHA-256 `{digest}`)" for path, digest in e["input_sha256"].items()],
               f"- Received: `{f['response']}` (response body as received, SHA-256 `{e['response_sha256']}`)"
               + (f"; `{f['output']}` (text of the response, unedited, SHA-256 `{e['output_sha256']}`)" if e["parsed"] else ""),
               ""]
    if lost:
        md += ["## Requests that returned no response", ""]
        md += [f"- {e['sent_utc']}: {e['error']}" + (f" (HTTP {e['http_status']})" if e["http_status"] else "")
               + f"; model `{e['model_requested']}`; parameters {json.dumps(e['parameters_sent'])}; "
               f"input text SHA-256 `{e['prompt_sha256']['input']}`. Message: {e['message']}"
               for e in lost]
        md.append("")
    save("RUN.md", "\n".join(md))

log_path = OUT_DIR / "run_log.jsonl"
entries = ([json.loads(line) for line in log_path.read_text(encoding="utf-8").splitlines() if line.strip()]
           if log_path.exists() else [])
run_no = sum(e["kind"] == "response" for e in entries) + 1
suffix = "" if run_no == 1 else f"_run{run_no}"
files = {"instructions": f"prompts/instructions{suffix}.txt", "input": f"prompts/input{suffix}.txt",
         "inputs": f"inputs{suffix}", "response": f"outputs/response_raw{suffix}.json", "output": f"outputs/output{suffix}.md"}

# Checks before anything is sent
if prompt_built_from != {"source": input_source, "sha256": sha256}:
    raise RuntimeError("The input files were loaded again after the prompt was built: run step 4 again. Nothing was sent.")
if run_no > 1 and not ALLOW_RERUN:
    raise RuntimeError(
        f"{run_no - 1} request(s) sent from this folder already returned a response (see {OUT_DIR / 'RUN.md'}). "
        "Nothing was sent. The manuscript reports a single run; to send another request on purpose, "
        "set ALLOW_RERUN = True in step 2 and run steps 2 to 5 again.")
leftover = [name for name in files.values() if (OUT_DIR / name).exists()]
if leftover:
    raise RuntimeError(f"Files that run {run_no} would write already exist in {OUT_DIR}: {leftover}. Nothing was sent. "
                       "Move the folder away, or check why they are there, before sending a request.")

print(f"Ready to send ONE request (run {run_no}): model {MODEL}, {len(USER_INPUT)} characters of input.")
if ASK_BEFORE_SENDING and input("Type SEND to send it; anything else cancels: ").strip() != "SEND":
    raise RuntimeError("Cancelled. Nothing was sent.")

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    try:
        from google.colab import userdata           # the "Secrets" panel of Colab, if the key is stored there
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        api_key = None
if not api_key:
    api_key = getpass.getpass("OpenAI API key: ")
api_key = api_key.strip()
if not api_key or not api_key.isascii() or not api_key.isprintable() or " " in api_key:
    raise ValueError("The API key is empty or contains spaces or unusual characters. Nothing was sent; enter it again.")

client = OpenAI(api_key=api_key, max_retries=0)     # no automatic retries: one request means one request
endpoint = str(client.base_url).rstrip("/") + "/responses"
request = {"model": MODEL, "instructions": INSTRUCTIONS, "input": USER_INPUT, "max_output_tokens": MAX_OUTPUT_TOKENS}
if REASONING_EFFORT is not None:
    request["reasoning"] = {"effort": REASONING_EFFORT}
if TEMPERATURE is not None:
    request["temperature"] = TEMPERATURE
parameters_sent = {k: v for k, v in request.items() if k not in ("model", "instructions", "input")}

prompt_sha256 = {"instructions": hashlib.sha256(INSTRUCTIONS.encode("utf-8")).hexdigest(),
                 "input": hashlib.sha256(USER_INPUT.encode("utf-8")).hexdigest()}

sent_at = datetime.datetime.now(datetime.timezone.utc)
try:
    http = client.responses.with_raw_response.create(**request)
except BaseException as error:              # also an interruption while waiting: the request may have been processed
    message = str(error).replace(api_key, "[API key]")
    entries.append({"kind": "no_response", "sent_utc": utc_stamp(sent_at), "endpoint": endpoint, "model_requested": MODEL,
                    "parameters_sent": parameters_sent, "prompt_sha256": prompt_sha256, "error": type(error).__name__,
                    "http_status": getattr(error, "status_code", None), "message": message[:1000] or "(no message)"})
    save_record(entries)
    if not isinstance(error, openai.OpenAIError):
        raise
    refused = isinstance(error, openai.APIStatusError)
    raise RuntimeError(
        f"The request returned no response ({type(error).__name__}): {message[:500]}\n"
        + ("The API refused it, so no text was generated. If the message names a parameter (for example temperature "
           "or reasoning effort), change it in step 2 (None = do not send it) and run steps 2 to 5 again."
           if refused else
           "The connection failed or timed out. The API may still have processed the request; check the usage page "
           "of the account before sending it again.")
        + f"\nThe attempt is recorded in {OUT_DIR / 'RUN.md'}.") from None

# A response arrived: write it to disk and log it before doing anything else with it
body = http.content
save(files["response"], body)
save(files["instructions"], INSTRUCTIONS)
save(files["input"], USER_INPUT)
for path, raw in raw_inputs.items():
    save(f"{files['inputs']}/{pathlib.Path(path).name}", raw)
entry = {
    "kind": "response", "run": run_no,
    "sent_utc": utc_stamp(sent_at),
    "sent_local": sent_at.astimezone(zoneinfo.ZoneInfo(LOCAL_TZ)).strftime("%Y-%m-%d %H:%M:%S ") + LOCAL_TZ,
    "endpoint": endpoint, "openai_python": openai.__version__, "python": platform.python_version(),
    "model_requested": MODEL, "parameters_sent": parameters_sent,
    "request_id": http.headers.get("x-request-id"),
    "input_source": prompt_built_from["source"], "input_sha256": prompt_built_from["sha256"],
    "prompt_sha256": prompt_sha256,
    "response_sha256": hashlib.sha256(body).hexdigest(),
    "files": files, "parsed": False, "usable": False,
}
entries.append(entry)
save_record(entries)

try:
    response = http.parse()
    output_text = response.output_text
    usage = response.usage
    entry.update({
        "model_returned": response.model, "response_id": response.id, "status": response.status,
        "incomplete_reason": getattr(response.incomplete_details, "reason", None),
        "settings_reported_by_api": {"temperature": response.temperature, "top_p": response.top_p,
                                     "reasoning effort": getattr(response.reasoning, "effort", None),
                                     "max_output_tokens": response.max_output_tokens,
                                     "number of tools": len(response.tools or [])},
        "tokens": None if usage is None else {
            "input": usage.input_tokens, "output": usage.output_tokens, "total": usage.total_tokens,
            "reasoning": getattr(getattr(usage, "output_tokens_details", None), "reasoning_tokens", None)},
        "output_sha256": hashlib.sha256(output_text.encode("utf-8")).hexdigest(),
        "usable": response.status == "completed" and bool(output_text.strip()),
        "parsed": True,
    })
except Exception as error:
    entry["note"] = f"the body could not be read as an API response ({type(error).__name__})"
    save_record(entries)
    shutil.make_archive("llm_run", "zip", root_dir=".", base_dir=str(OUT_DIR))
    raise RuntimeError(f"A response arrived and was saved as {OUT_DIR / files['response']}, but it could not be read "
                       f"({type(error).__name__}). It is recorded as run {run_no}; look at the saved file.") from None
save(files["output"], output_text)
save_record(entries)
shutil.make_archive("llm_run", "zip", root_dir=".", base_dir=str(OUT_DIR))

print(f"run {run_no}: model {entry['model_returned']}, status {entry['status']}, "
      f"{'no usage reported' if usage is None else str(usage.output_tokens) + ' output tokens'}, {entry['sent_local']}")
if not entry["usable"]:
    raise RuntimeError(
        f"The response has no usable output (status {entry['status']}, reason {entry['incomplete_reason']}, "
        f"{len(output_text)} characters of text). It was saved and recorded as run {run_no}; step 6 shows the record. "
        "If the output limit was reached, raise MAX_OUTPUT_TOKENS, set ALLOW_RERUN = True and run steps 2 to 5 again; "
        "Section 2.9 must then mention that the first request was cut off.")

## 6. Output

The cell prints the unedited model output and the run record, and downloads `llm_run.zip`. The archive holds
the folder `llm/`; unpack it at the top level of the repository, so that its files go into the existing `llm/`
folder, and add this notebook to `llm/code/`. Before publishing the notebook, check that no output cell shows
an API key (clearing all outputs removes any doubt).

In [ ]:
# 6. Output and download
record = OUT_DIR / "run_log.jsonl"
if not record.exists():
    raise RuntimeError("No request has been sent from this folder yet: run step 5 first.")
logged = [json.loads(line) for line in record.read_text(encoding="utf-8").splitlines() if line.strip()]
usable = [e for e in logged if e["kind"] == "response" and e["usable"]]
if usable:
    print(f"--- model output of run {usable[-1]['run']} ({usable[-1]['files']['output']}) ---\n")
    print((OUT_DIR / usable[-1]["files"]["output"]).read_text(encoding="utf-8"))
else:
    print("There is no usable model output yet.")
print("\n" + "=" * 70 + "\n")
print((OUT_DIR / "RUN.md").read_text(encoding="utf-8"))

archive = shutil.make_archive("llm_run", "zip", root_dir=".", base_dir=str(OUT_DIR))
try:
    from google.colab import files as colab_files
    colab_files.download(archive)
except ImportError:
    pass
print("archive:", archive)